# Homework 03 - Value Source Reading

目标：用具体变量代入理解 `__add__`、`__mul__`、闭包、`backward()`。

In [ ]:
from pathlib import Path
import sys, math, random


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

from course.checks import qa_check

## 完整例子 - 把 Python 调用代入具体变量

当你写 `c = a + b`，Python 会调用 `a.__add__(b)`。所以 `self=a`，`other=b`，返回的 `out` 就是 `c`。

In [ ]:
from micrograd.engine import Value

a = Value(2.0)
b = Value(3.0)
c = a + b
print('c.data =', c.data)
print('c._op =', c._op)
print('c._prev size =', len(c._prev))

## Modify - 换成 `c = a * 3`

这次 `other` 一开始是普通数字 `3`，源码会把它包装成 `Value(3)`。

In [ ]:
# 填写说明：
# - 完成 student_mul_probe，用运行结果观察 a * 3 的前向建图。
# - 返回顺序固定：out.data、out._op、len(out._prev)。
# - 运行本 cell，看 qa_check 的提示。

from micrograd.engine import Value


def student_mul_probe():
    a = Value(2.0)
    out = a * 3
    # TODO: return out_data, out_op, parent_count
    pass


qa_check('qa_check_03_modify', globals())


<details><summary>Show / Hide 本题提示</summary>

- `a * 3` 能跑，是因为源码会把普通数字 `3` 包装成 `Value(3)`。
- 包装后它和 `a * b` 一样，都是两个父节点产生一个乘法输出节点。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_mul_probe():
    a = Value(2.0)
    out = a * 3
    return out.data, out._op, len(out._prev)
```

</details>


In [ ]:
# 填写说明：
# - 这一组拆成两个小函数：先观察 a+b，再观察 a+3。
# - 第二个函数和第一个函数形状相同，只是把 b 换成普通数字 3。
# - 每个函数返回顺序固定：out.data、out._op、len(out._prev)。
# - 运行本 cell，看 qa_check 的提示。

from micrograd.engine import Value


def student_add_value_probe():
    a = Value(2.0)
    b = Value(3.0)
    c = a + b
    # TODO: return c_data, c_op, c_parent_count
    pass


def student_add_number_probe():
    # TODO: 参考 student_add_value_probe，只把 b 换成普通数字 3。
    pass


def student_add_probe():
    # TODO: 调用前两个函数，return value_probe, number_probe
    pass


qa_check('qa_check_source_reading_basic', globals())


<details><summary>Show / Hide 本题提示</summary>

- `c = a + b` 和 `d = a + 3` 都会产生新的输出节点。
- 如果 `3` 被包装成 `Value(3)`，那么 `d` 也应该有两个父节点。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_add_value_probe():
    a = Value(2.0)
    b = Value(3.0)
    c = a + b
    return c.data, c._op, len(c._prev)


def student_add_number_probe():
    a = Value(2.0)
    d = a + 3
    return d.data, d._op, len(d._prev)


def student_add_probe():
    return student_add_value_probe(), student_add_number_probe()
```

</details>


In [ ]:
# 填写说明：
# - 完成 student_backward_probe，用 L=a+a 观察 += 和 L.grad=1 的效果。
# - 返回顺序固定：a.grad、L.grad。
# - 运行本 cell，看 qa_check 的提示。

from micrograd.engine import Value


def student_backward_probe():
    a = Value(2.0)
    L = a + a
    # TODO 1: 调用 backward
    # TODO 2: return a_grad, L_grad
    pass


qa_check('qa_check_source_reading_backward', globals())


<details><summary>Show / Hide 本题提示</summary>

- `L = a + a` 里，同一个 `a` 走了两条边到 `L`。
- `L.grad` 表示最终输出对自己的梯度，应该从 1 开始。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_backward_probe():
    a = Value(2.0)
    L = a + a
    L.backward()
    return a.grad, L.grad
```

</details>


In [ ]:
# 填写说明：
# - 完成 student_build_topo_demo，修复拓扑排序的两个常见语法坑。
# - 运行本 cell，看 qa_check 的提示。

# Debug Lab：你之前踩过两个坑：for 循环写法、append 写成 add。
def student_build_topo_demo(root):
    topo = []
    visited = set()

    def build_topo(v):
        # TODO: 如果访问过就返回；否则先递归父节点，最后 append 自己。
        pass

    build_topo(root)
    return topo


qa_check('qa_check_source_debug', globals())


<details><summary>Show / Hide 本题提示</summary>

- 遍历父节点的写法是 `for child in v._prev:`。
- `topo` 是 list，最后加入节点用 `topo.append(v)`。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_build_topo_demo(root):
    topo = []
    visited = set()

    def build_topo(v):
        if v in visited:
            return
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)

    build_topo(root)
    return topo
```

</details>


<details><summary>Show / Hide 提示</summary>

先找完整例子的形状，再只改一个地方。填 TODO 前，先用小数字在纸上算一遍；测试失败时先判断错在数学、Python、计算图还是训练循环。

</details>

<details><summary>Show / Hide 答案</summary>

答案不要一开始看。正确答案应该能由完整例子和 Modify 题一步推出；如果你需要直接看答案，说明前一个台阶还没踩稳。

</details>

## 提交清单

- [ ] 我跑过完整例子。
- [ ] 我完成了 Modify 题。
- [ ] 我完成了 TODO 作业并通过 qa_check。
- [ ] 我修过 Debug Lab。
- [ ] 我能说出本节知识如何接回 micrograd 主线。